#### CARGA INCREMENTAL

CONFIGURACION

In [0]:

CATALOG = "pentaho_logs"
SCHEMA = "bronze"
TABLE = "bronze_logs_pentaho"
VOLUME_PATH = "/Volumes/pentaho_logs/bronze/volume_pentahologs"
TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE}"

#### OBTENCION DE NOMBRE POR VOLUMEN

In [0]:
if dbutils.fs.ls(VOLUME_PATH):

    archivos_volumen = [
        archivo.name
        for archivo in dbutils.fs.ls(VOLUME_PATH)
        if archivo.name.startswith("pentaho.log.")
    ]
    for archivo in archivos_volumen:
        print(archivo)

    display(
        spark.createDataFrame(
            [(archivo,) for archivo in archivos_volumen],
            ["file_name"]
        )
    )

else:
    print("No hay archivos en el Volume")
    archivos_volumen = []

#### OBTENCIÓN DE NOMBRE POR TABLA


In [0]:
if spark.catalog.tableExists(TABLE_NAME):
 print("La tabla bronze existe")
 archivos_bronze = [
       row.file_name
        for row in (spark.table(TABLE_NAME).select("file_name").distinct().collect())
    ]
 
 print("=== ARCHIVOS EN BRONZE ===")
 for archivo in archivos_bronze:
   print(archivo)
else:
 archivos_bronze = []

#### COMPARAR VOLUMEN VS TABLA BRONZE

In [0]:
## comparamos los archivos en un arreglo si hay archivos en donde sean diferentes ingresa

archivos_nuevos = [
    archivo
    for archivo in archivos_volumen
    if archivo not in archivos_bronze
]

print("=== NUEVOS ===")
print(archivos_nuevos)

#### Envio parametro capa Bronze

In [0]:
import json 

if archivos_nuevos:
    archivos_nuevos_json =json.dumps(archivos_nuevos)
    resultado = dbutils.notebook.run("/Workspace/Pentaho_logs_Intelligence/bronze/bronze_pentaho/03_Bronze_Ingestion_Pentaho_log",
                                     0,
                                     {
                                         "archivos_nuevos" : archivos_nuevos_json
                                      })
    print(resultado)

else:
    print("No hay archivos nuevos para procesar")


    

#### Envio parametro capa Silver

In [0]:
import json 

if archivos_nuevos:
    archivos_nuevos_json =json.dumps(archivos_nuevos)
    resultado = dbutils.notebook.run("/Workspace/Pentaho_logs_Intelligence/silver/silver_pentaho/03_Silver_Transformation_Pentaho_Log",
                                     0,
                                     {
                                         "archivos_nuevos" : archivos_nuevos_json
                                      })
    print(resultado)

else:
    print("No hay archivos nuevos para procesar")



#### Envio parametro capa Gold

In [0]:
import json 

if archivos_nuevos:
    archivos_nuevos_json =json.dumps(archivos_nuevos)
    resultado = dbutils.notebook.run("/Workspace/Pentaho_logs_Intelligence/gold/gold_pentaho/03_Gold_Pentaho_Log",
                                     0,
                                     {
                                         "archivos_nuevos" : archivos_nuevos_json
                                      })
    print(resultado)

else:
    print("No hay archivos nuevos para procesar")